In [149]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Created on 2024-05-27

@author: Juan Enrique López

@description: Jupyter Notebook creado para leer la información publicada en diferentes portales web de noticias sobre hacking y descargar la información
de aquellas que puedan relacionarse por TTP

"""

'\nCreated on 2024-05-27\n\n@author: Juan Enrique López\n\n@description: Jupyter Notebook creado para leer la información publicada en diferentes portales web de noticias sobre hacking y descargar la información\nde aquellas que puedan relacionarse por TTP\n\n'

In [150]:
import requests
from bs4 import BeautifulSoup

from datetime import datetime
from attackcti import attack_client
import pandas as pd
import yaml
import os
import re
from dateutil.parser import parse


"""
Importante aumentar la recursividad para alcanzar la serialización
"""
# import sys
# sys.setrecursionlimit(10 ** 6)

'\nImportante aumentar la recursividad para alcanzar la serialización\n'

In [151]:
#https://thehackernews.com/
#https://www.bleepingcomputer.com/

#### **Parámetros**

In [152]:
'''
Importante, se va a utilizar el método range, por lo que será contar desde 0 hasta el valor definido. Por ejemplo, si num_pages es 2 contará 2 url posteriores a la página principal, por lo que dispondremos de un total de 3 hojas de contenido.
'''
num_pages = 5
url = 'https://thehackernews.com/'

In [153]:
# Parámetro elección tipo guardado
save_by_category_and_subcategory = False
save_by_unified_category = False
save_by_ttp = False
save_by_date = True

# Guardado por tipo de extensión
save_as_yaml = False
save_as_md = True

#### **Funciones**

In [154]:
def link_to_soup(link):
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }
    response = requests.get(link, headers=headers)
    if response.ok:
        return BeautifulSoup(response.text, 'html.parser')
    else:
        return False

In [155]:
def get_news_urls(main_bs4_page, num_scrap_pages, class_searched="blog-pager-older-link-mobile"):
    pages = []
    pages.append(main_bs4_page)
    urls_temp = [url]
    posts_in_page = []
    urls_news = []
    check_url = r'https://thehackernews.com/202'

    # Mediante un bucle recorremos el número de paginas solicitadas para obtener los links de navegación entre páginas. Añadimos el contenido de toda la página al objeto pages
    for item in range(num_scrap_pages):
        # Buscamos, comenzando por la home page el link que te lleva a la siguiente url con noticias
        next_page_link = pages[item].find("a", class_=class_searched)['href']
        # Añadimos la url en una lista para tenerlo controlado
        urls_temp.append(next_page_link)
        # Generamos el objeto bs4 y lo añadimos
        pages.append(link_to_soup(next_page_link))
    # Filtramos el contenido con el objetivo de obtener las url de las noticias
    for page in pages:
        # Buscamos dentro del objeto bs4 la clase utilizada para publicar las noticias, importante que esto puede utilizarse para alojar anuncios y debemos hacer un filtro adicional
        posts_in_page = page.find_all("a", class_='story-link')
        for post in posts_in_page:
            # Filtramos solamente url que contengan la subcadena que las identifica como noticias
            if check_url.lower() in post['href'].lower():
                urls_news.append(post['href'])

    # Almacenadas las url que se van a scrapear
    urls_temp
    # Almacenado en un objeto todo el contenido de cada una de las urls
    pages
    return urls_news

In [156]:
def get_date_from_news(news):
    # Obtener la fecha de publicación de la noticia
    spans = news.find_all('span')
    date_str = []
    # Iterar sobre los elementos <span> para encontrar aquellos con la clase "author"
    for span in spans:
        # Obtener la lista de clases del elemento <span>
        span_classes = span.get('class', [])
        # Verificar si "author" está en la lista de clases
        if 'author' in span_classes:
            date_str.append(span.get_text())
    try:
        # date_news = datetime.strptime(date_str[0], '%B %d, %Y')
        date_news = parse(date_str[0])
    except Exception as e:
        print(f'Error al intentar parsear la fecha: {e}')
    date_news = date_news.strftime('%d/%m/%Y')
    return date_news

In [157]:
def get_cat_from_news(news):
    # Obtener la fecha de publicación de la noticia
    spans = news.find_all('span')
    cat_str = []
    # Iterar sobre los elementos <span> para encontrar aquellos con la clase "p-tags"
    try:
        for span in spans:
            # Obtener la lista de clases del elemento <span>
            span_classes = span.get('class', [])
            # Verificar si "p-tags" está en la lista de clases
            if 'p-tags' in span_classes:
                cat_str.append(str(span.get_text()).replace('_', ' '))
    except:
        cat_str=''
    
    return cat_str

In [158]:
def get_body_and_links_from_news(news):
    body = []
    news_links = []
    for p in news.find_all('p'):
        body.append(str(p.get_text()))
        links = p.find_all('a')
        for link in links:
            news_links.append(str(link.get('href')))
    return news_links, body

In [159]:
# Función que, dada una lista de urls que apuntan a las noticias de TheHackersNews, retorna un diccionario con la información
def get_news_info(news_url_list):
    news_info = []
    for n in news_url_list:
        n_info = {}
        n_bs4 = link_to_soup(n)
        n_title = str(n_bs4.title.string)
        print(n)
        n_date = get_date_from_news(n_bs4)
        n_cat = get_cat_from_news(n_bs4)
        n_links, n_body = get_body_and_links_from_news(n_bs4)
        
        n_info['title'] = n_title
        n_info['date'] = n_date
        n_info['url_news'] = n
        n_info['thn_category_and_subcategory'] = n_cat
        n_info['links'] = n_links
        n_info['body'] = n_body
        news_info.append(n_info)

    return news_info

In [160]:
# Función generalista de formateo que elimina caracteres extraños
def format_name(item):
    # Convertimos en string en el caso de pasar una lista
    if isinstance(item, list):
        item = '_'.join(map(str, item))
    # Eliminamos caracteres no válidos usando una expresión regular
    item = re.sub(r'[\\/:*?"<>|]', '', item)
    item = re.sub(r'\s+', '_', item)  # Reemplazar espacios por guiones bajos
    item = re.sub(r'[^a-zA-Z0-9_\-]', '', item)  # Eliminar cualquier otro carácter no alfanumérico
    return item

In [161]:
# Función específica para formatear la categoría obtenida 
def format_name_cat(item):
    # Convertimos en string en el caso de pasar una lista
    if isinstance(item, list):
        item = '_'.join(map(str, item))
    # En primer lugar nos asguramos de que no exista ninguna barra baja
    item = item.replace('_', ' ')
    # Reemplazamos el caracter que separa categoria de dsubcategoria por barra baja
    item = item.replace(' / ', '_')
    # Reemplazar espacios por guiones
    item = item.replace(' ', '-') 
    # Reemplazamos resto de caracteres no alfanumérico
    item = re.sub(r'[^a-zA-Z0-9_\-]', '', item)
    return item

In [162]:
# Función especifica de formateo para la unificación de las categorías
def split_list_by_hyphen(items):
    result = []
    for item in items:
        item = item.replace('-', ' ')
        if '/' in item:
            parts = item.split('/')
            result.extend([part.strip() for part in parts])  # Elimina espacios en blanco alrededor
        else:
            result.append(item.strip())  # Elimina espacios en blanco alrededor
    return result

In [163]:
# Función específica para unificar la categoría obtenida 
def get_unified_cat(item):
    cats = split_list_by_hyphen(item)
    cats = [elemento.strip() for elemento in cats]
    cats = [re.sub(r'[^a-zA-Z0-9_\-, ]', '', cat) for cat in cats]
    return cats

In [164]:
def create_header_properties(dictionary):
    cat_subcat = dictionary.get('thn_category_and_subcategory', [])[:1] or []
    cat_subcat = [subitem.strip() for str in cat_subcat for subitem in str.split('/')]
    # cat_subcat = [cat.strip() for cat in cat_subcat]
    cat, subcat = (cat_subcat + ['', ''])[:2]
    
    header = f"""---
CP Source: The Hacker News
CP Execution date:  {datetime.now().strftime('%Y-%m-%d')}
Headline: {'"' + dictionary.get('title') + '"'}
Date: {datetime.strptime(dictionary.get('date'), '%d/%m/%Y').strftime('%Y-%m-%d')}
Category: {'"'+cat+'"'}
Sub-Category: {'"'+subcat+'"'}
---
"""
    return header

In [165]:
# Subclase personalizada de yaml.SafeDumper que evita las referencias alias.
# Esto puede ayudar a evitar problemas con estructuras de datos complejas que PyYAML no maneja bien de forma predeterminada.
class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

# Función de escritura del yaml para TheHackersNews
# Deprecated
def write_dict_to_yaml(news, filename):
    try:
        # Ordenamos las claves del diccionario
        ordered_keys = ['title', 'url_news', 'thn_category_and_subcategory', 'date', 'links', 'body']
        ordered_news = {key: news[key] for key in ordered_keys if key in news}
        
        with open(filename, 'w') as file:
            yaml.dump(ordered_news, file, default_flow_style=False, allow_unicode=True, Dumper=NoAliasDumper, sort_keys=False)
        print(f"Noticia escrita en {filename}")
    except yaml.YAMLError as e:
        print(f"Error al serializar el archivo YAML: {e}")
    except IOError as e:
        print(f"Error de I/O al escribir el archivo: {e}")

In [166]:
def write_dict_to_md(dictionary, file_path):
    try:
        with open(file_path, 'w', encoding='utf-8') as file:
            header_properties = create_header_properties(dictionary)
            file.write(header_properties+ "\n")
            for key, value in dictionary.items():
                value = str(value).replace('[', '').replace(']', '')
                file.write(f"**{key}**\n{value}\n\n")
        print(f"Archivo .md guardado en: {file_path}")
    except (OSError, IOError) as e:
        print(f"Error al escribir en el archivo {file_path}: {e}")

In [167]:
# Función que lista los archivos ubicados en un directorio y subdirectorios
def get_list_files_subdirectory(dirName):
    # Crea lista de ficheros y subdirectorios
    listOfFile = os.listdir(dirName)
    allFiles = list()
    #Recorre los directorios y subdirectorios y genera una lista con los ficheros encontrados en estos
    for entry in listOfFile:
        fullPath = os.path.join(dirName, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_files_subdirectory(fullPath)
        else:
            allFiles.append(fullPath)
    #Devuelve la lista con los ficheros
    return allFiles

In [168]:
def print_oldest_and_most_recent_notice(dict_news):
    from datetime import datetime
    dates = [datetime.strptime(dic['date'], '%d/%m/%Y') for dic in dict_news]
    oldest = min(dates)
    most_recent = max(dates)
    oldest = oldest.strftime('%d/%m/%Y')
    most_recent = most_recent.strftime('%d/%m/%Y')
    print('Noticia más reciente: '+most_recent)
    print('Noticia más antigua: '+oldest)

In [169]:
def get_count_files_from_path(main_folder,col_name):
    name_folders = []
    count_files = []
    # Recorrer todas las carpetas y archivos en la carpeta principal
    for path, subcarpetas, archivos in os.walk(main_folder):
        if path == main_folder:
            continue
        name_folder = os.path.basename(path)
        # Contar el número de archivos en la carpeta actual
        count_file = len(archivos)
        # Agregar el nombre de la carpeta y el conteo a las listas
        name_folders.append(name_folder)
        count_files.append(count_file)
    # Crear un DataFrame de pandas
    df = pd.DataFrame({col_name: name_folders, 'items': count_files})
    # df['cat_subcat'] = df['cat_subcat'].apply(lambda x: x if '_' in x else f"{x}_")
    return df

In [170]:
# Pequeña función para formatear posibles casos en los que se hayan asignado más de una subcategoría 
def modify_string(s):
    if s.count('_') == 2:
        # Reemplazar el segundo "_" encontrado
        parts = s.split('_', 2)
        return parts[0] + '_' + parts[1] + parts[2]
    return s

In [171]:
# Llamada a las técnicas y obtención de la lista
def techniques():
    lift = attack_client()
    #Solicitud a la libreria attackcti la extraccion de las tecnicas Enterprise y normalizar el json en un dataframe
    techniques = lift.get_enterprise_techniques(stix_format=False)
    techniques = pd.json_normalize(techniques)
    #Eliminar las tecnicas deprecadas y revocadas
    techniques = techniques[(techniques['mitre_deprecated'] != True)]
    # Eliminamos duplicados y convertimos en lista
    techniques = techniques['technique_id'].drop_duplicates().tolist()
    return techniques
techniques_enterprise = techniques()
techniques_enterprise = sorted(techniques_enterprise, key=len, reverse=True)
techniques_enterprise[0:3]

# Si el server está caido:
# path_techniques = r'C:\Users\jelopez\Documents\CyberProof\python\develop\resources_attackcti_down/id_tecnicas_16052024.csv'
# techniques_enterprise = pd.read_csv(path_techniques)
# techniques_enterprise = techniques_enterprise['ID Tecnica'].to_list()
# techniques_enterprise = sorted(techniques_enterprise, key=len, reverse=True)
# techniques_enterprise[:3]

[taxii2client.v20] [WARNING ] [2024-06-21 11:55:35,580] TAXII Server Response did not include 'Content-Range' header - results could be incomplete.
[taxii2client.v20] [WARNING ] [2024-06-21 11:55:35,612] TAXII Server Response with different amount of objects! Setting per_request=780


['T1059.010', 'T1564.012', 'T1027.013']

In [172]:
# Función encargada de buscar en un texto 
def find_techniques(texto):
    if isinstance(texto, str):
        found = []
        for item in techniques_enterprise:
            if item in texto:
                found.append(item)
        return ', '.join(found)
    else:
        return ''

#### **Ejecución principal**

In [173]:
# Establecemos la página principal en un objeto de bs4
home_page = link_to_soup(url)

In [174]:
# Obtención de las url de noticias 
news_list = get_news_urls(home_page, num_pages, class_searched="blog-pager-older-link-mobile")
print(f'Se han obtenido {str(len(news_list))} noticias')

Se han obtenido 60 noticias


In [175]:
# Obtenemos la información de las noticias y la almacenamos en una lista de diccionarios
news_info = get_news_info(news_list)

https://thehackernews.com/2024/06/oyster-backdoor-spreading-via.html
https://thehackernews.com/2024/06/solarwinds-serv-u-vulnerability-under.html
https://thehackernews.com/2024/06/us-bans-kaspersky-software-citing.html
https://thehackernews.com/2024/06/researchers-uncover-uefi-vulnerability.html
https://thehackernews.com/2024/06/french-diplomatic-entities-targeted-in.html
https://thehackernews.com/2024/06/tool-overload-why-msps-are-still.html
https://thehackernews.com/2024/06/chinese-cyber-espionage-targets-telecom.html
https://thehackernews.com/2024/06/cybersecurity-cpes-unraveling-what-why.html
https://thehackernews.com/2024/06/new-rust-based-fickle-malware-uses.html
https://thehackernews.com/2024/06/experts-uncover-new-evasive-squidloader.html
https://thehackernews.com/2024/06/kraken-crypto-exchange-hit-by-3-million.html
https://thehackernews.com/2024/06/chinese-cyber-espionage-group-exploits.html
https://thehackernews.com/2024/06/new-case-study-unmanaged-gtm-tags.html
https://theha

**Guardado de los archivos por categoría y subcategoría THN**

In [176]:
if save_by_category_and_subcategory:
    for n in news_info:
        if len(n.get('thn_category_and_subcategory'))==1:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews','save_by_category_and_subcategory', format_name_cat(n.get('thn_category_and_subcategory')).lower())
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_category_and_subcategory', 'no-category')
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        if save_as_yaml:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
            write_dict_to_yaml(n, filename)
        elif save_as_md:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
            write_dict_to_md(n, filename)
    print_oldest_and_most_recent_notice(news_info)

**Revisión resultados - categoría y subcategoría THN**

In [177]:
if save_by_category_and_subcategory:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_category_and_subcategory')
    result = get_count_files_from_path(main_folder, 'cat_subcat')
    result = result.sort_values(by='items', ascending=False)
    #Filtramos la carpeta outputs
    result = result[result['cat_subcat']!='outputs']
    result['cat_subcat'] = result['cat_subcat'].apply(modify_string)
    # # Añadimos una barra baja al final de todos los string que no contengan subcategoria
    result['cat_subcat'] = result['cat_subcat'].apply(lambda x: x if '_' in x else f"{x}_")
    result[['cat', 'subcat']] = result['cat_subcat'].str.split('_', expand=True)
    result.sort_values(by='items', ascending=False).head(5)

**Guardado de los archivos por categoría unificada THN**

In [178]:
if save_by_unified_category:
    for n in news_info:
        if len(n.get('thn_category_and_subcategory'))==1:
            cats = get_unified_cat(n.get('thn_category_and_subcategory'))
            for cat in cats:
                folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_unified_category', cat.lower())
                if not os.path.exists(folder_name):
                    os.makedirs(folder_name)
                if save_as_yaml:
                    filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
                    write_dict_to_yaml(n, filename)
                elif save_as_md:
                    filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
                    write_dict_to_md(n, filename)
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_unified_category', 'no-category')
            if not os.path.exists(folder_name):
                os.makedirs(folder_name)
            if save_as_yaml:
                filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
                write_dict_to_yaml(n, filename)
            elif save_as_md:
                filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
                write_dict_to_md(n, filename)
    print_oldest_and_most_recent_notice(news_info)
    

**Revisión resultados - categoría unificada THN**

In [179]:
if save_by_unified_category:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_unified_category')
    results = get_count_files_from_path(main_folder, 'save_by_unified_category') # Chequear la introduccion de la subcarpeta 'TheHackerNews',
    display(results.sort_values(by='items', ascending=False).head(10))

**Búsqueda de TTP's en la noticia**

In [180]:
if save_by_ttp:
    ttp_list = []
    for n in news_info:
        unified_body = ' '.join(n['body'])
        ttp_finded = find_techniques(unified_body)
        ttp_list.append(ttp_finded)
ttp_list_unique_set = set(ttp_list)
ttp_list_unique = list(ttp_list_unique_set)
# TTP's unicas encontradas
ttp_list_unique

['']

**Guardado por TTP en la noticia**

In [181]:
if save_by_ttp and len(ttp_list_unique) > 1:
    for n in news_info:
        if len(n.get('thn_category_and_subcategory'))==1:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_ttp', format_name_cat(n.get('thn_category_and_subcategory')).lower())
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'no-category')
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        if save_as_yaml:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
            write_dict_to_yaml(n, filename)
        elif save_as_md:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
            write_dict_to_md(n, filename)
elif len(ttp_list_unique) <= 1:
    print("No se han obtenido TTP's de las noticias por lo que no se ha procedido al guardado.")

No se han obtenido TTP's de las noticias por lo que no se ha procedido al guardado.


**Guardado por fecha**

In [182]:
if save_by_date:
    for n in news_info:
        if len(n.get('date')) != '':
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_date', datetime.strptime(n.get('date'), '%d/%m/%Y').strftime('%Y%m%d'))
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_date', 'no-date')
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        if save_as_yaml:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
            write_dict_to_yaml(n, filename)
        elif save_as_md:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
            write_dict_to_md(n, filename)

Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\web_scraping\outputs\TheHackerNews\save_by_date\20240621\oyster_backdoor_spreading_via_trojanized_popular_software_downloads.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\web_scraping\outputs\TheHackerNews\save_by_date\20240621\solarwinds_serv-u_vulnerability_under_active_attack_-_patch_immediately.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\web_scraping\outputs\TheHackerNews\save_by_date\20240621\us_bans_kaspersky_software_citing_national_security_risks.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\web_scraping\outputs\TheHackerNews\save_by_date\20240620\researchers_uncover_uefi_vulnerability_affecting_multiple_intel_cpus.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\web_scraping\outputs\TheHackerNews\save_by_date\20240620\french_diplomatic_entities_targeted_in_russian

****Revisión resultados - fecha de la noticia****

In [183]:
if save_by_date:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'TheHackerNews', 'save_by_date')
    results = get_count_files_from_path(main_folder, 'date')
    display(results.sort_values(by='items', ascending=False).head(10))

,date,items
3,20240613,8
2,20240612,6
4,20240614,6
8,20240618,6
9,20240619,6
10,20240620,6
7,20240617,5
1,20240611,4
5,20240615,3
11,20240621,3


#### **En desarrollo posible mejora:  buscar grupos dentro de las noticias**

In [184]:
# lift = attack_client()
# groups = lift.get_groups(stix_format=False)
# groups = pd.json_normalize(groups)
# groups = groups[['group', 'group_aliases', 'contributors']]
# groups['group_aliases'] = groups['group_aliases'].apply(lambda x: ', '.join(map(str, x)) if x is not None else '')
# groups['contributors'] = groups['contributors'].apply(lambda x: ', '.join(map(str, x)) if x is not None else '')
#groups



# test_list = groups.values.flatten().tolist()
# len(test_list)

# def groups():
#     lift = attack_client()
#     #Solicitud a la libreria attackcti la extraccion de las tecnicas Enterprise y normalizar el json en un dataframe
#     groups = lift.get_groups(stix_format=False)
#     techniques = pd.json_normalize(techniques)
#     #Eliminar las tecnicas deprecadas y revocadas
#     techniques = techniques[(techniques['mitre_deprecated'] != True)]
#     # Eliminamos duplicados y convertimos en lista
#     techniques = techniques['technique_id'].drop_duplicates().tolist()
#     return techniques
# techniques_enterprise = techniques()
# techniques_enterprise = sorted(techniques_enterprise, key=len, reverse=True)
# techniques_enterprise[0:3]


# def find_groups(texto):
#     if isinstance(texto, str):
#         found = []
#         for item in techniques_enterprise:
#             if item in texto:
#                 found.append(item)
#         return ', '.join(found)
#     else:
#         return ''